# Does stimulation move activity along the natural disengagement axis?

**Hypothesis.** Train a population coding direction (CD) to separate *engaged*
from *no-response* (and *reward* from *no-reward*) trials using **unstimulated**
data, then project **stimulation** trials onto that axis. If stimulation pushes
the population toward the natural *no-response* (disengaged) state — and the
per-trial magnitude predicts behaviour — that supports a causal role for the
stimulated region in disengagement.

**Design.**
1. Select units: all default-QC units, optionally restricted to **opto-tagged**
   units and/or specific **probes** (we have no anatomy yet, so probe/tag is the
   proxy for region).
2. Build the CD on **unstimulated** trials only, for two contrasts:
   - `response`: engaged (`animal_response != 2`) vs no-response (`== 2`)
   - `reward`:   rewarded vs unrewarded (responded, no reward)
3. Enumerate the **stimulation conditions** present in the session.
4. Project a chosen condition's trials onto the unstim axis and test whether
   they shift toward the disengaged pole; quantify with a **disengagement index**
   (0 = engaged pole, 1 = disengaged pole) and ask whether the per-trial
   projection predicts no-response behaviour.

The heavy lifting lives in `stim_axis_projection.py`; this notebook just wires
it together so it runs standalone on the capsule.

In [ ]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")

In [ ]:
# =============================================================================
# 2. CONFIG
# =============================================================================
import re
from pathlib import Path

# --- Session identifiers -----------------------------------------------------
session_date = '863973_2026-08-25_14-18-29'
ephys_session = None   # None -> auto-discover from the PSTH Zarr in step 3

# --- Paths -------------------------------------------------------------------
PSTH_DIR = Path("/root/capsule/scratch/psth_results")
RESULTS_DIR = Path("/root/capsule/scratch/behavior_summary")

# --- Unit selection (same combinable filters as the PSTH notebooks) ----------
#   USE_TAGGED : if True, keep only opto-tagged units (from the metrics CSV below)
#   PROBES     : if a non-empty list, keep only units on those NWB 'device_name'
#                probes (e.g. ["ProbeA"]); None/[] = all probes.
# If BOTH are set, a unit must satisfy BOTH (tagged AND on a listed probe).
USE_TAGGED = False     # True -> restrict to opto-tagged units
PROBES = None          # e.g. ["ProbeA"] or ["ProbeA", "ProbeB"]; None/[] -> all probes

# Opto-tagging source + condition (only used when USE_TAGGED is True).
metrics_csv_path = Path(f"/root/capsule/scratch/opto_tagging/metrics_{session_date}.csv")
#   (target_power, location_tag, laser_name, cycle_duration, frequency, pulse_duration)
target_cond = (5.0, 1.0, "Laser_1", 1.0, 10.0, 0.05)
SELECT_PULSE_INDEX = -1   # -1 = pooled over all pulses in the train

# --- Coding-direction parameters ---------------------------------------------
ALIGN_TO = "go_cue"       # alignment event for the CD
FIT_WINDOW = (0.0, 0.5)   # window (s) used to FIT the CD axis
PROJ_WINDOW = (-1.0, 2.0) # window (s) for the time-resolved projection trace (None = full)
AXES_TO_BUILD = ["response", "reward"]

# Output folder encodes the unit selection so runs don't overwrite each other.
_tag_part = "tagged" if USE_TAGGED else "allqc"
_probe_part = "all_probes" if not PROBES else "probe_" + "_".join(
    re.sub(r"[^0-9A-Za-z._-]", "", str(p)) for p in PROBES
)
OUTDIR = Path(f"/root/capsule/scratch/stim_cd_projection/{session_date}/{_tag_part}__{_probe_part}")

print(f"Use tagged : {USE_TAGGED}")
print(f"Probes     : {'all' if not PROBES else PROBES}")
print(f"Axes       : {AXES_TO_BUILD}")
print(f"Output dir : {OUTDIR}")

In [ ]:
# =============================================================================
# 3. LOAD SESSION RESOURCES (Zarr PSTH, behavior summary, NWB, trials)
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from general_utils import smart_read_csv
from behavior_utils import find_trials
from nwb_utils import NWBUtils
from create_psth import load_zarr

# Auto-discover the full sorted session name from the PSTH Zarr if not set.
if ephys_session is None:
    cands = sorted(PSTH_DIR.glob(f"ecephys_{session_date}_sorted*_0.2s.zarr"))
    if not cands:
        raise FileNotFoundError(
            f"No PSTH Zarr matching ecephys_{session_date}_sorted*_0.2s.zarr in {PSTH_DIR}"
        )
    ephys_session = cands[0].name[: -len("_0.2s.zarr")]
print(f"ephys_session: {ephys_session}")

zarr_path = PSTH_DIR / f"{ephys_session}_0.2s.zarr"
beh_csv = RESULTS_DIR / f"behavior_summary-{ephys_session}.csv"
assert zarr_path.exists(), f"Zarr not found: {zarr_path}"
assert beh_csv.exists(), f"Behavior CSV not found: {beh_csv}"

nwb, _ = NWBUtils.combine_nwb(session_name=ephys_session)
psth = load_zarr(str(zarr_path))
beh = smart_read_csv(str(beh_csv))

# Laser-on trials -> the stimulation set; everything else is 'unstimulated'.
if "laser_on_trial" in nwb.trials.colnames:
    laser_on = np.asarray(nwb.trials["laser_on_trial"][:])
    opto_trial_ids = np.where(laser_on == 1)[0].astype(np.int64)
else:
    opto_trial_ids = np.array([], dtype=np.int64)
print(f"laser-on (stimulation) trials: {len(opto_trial_ids)}")
print(f"response trials: {len(find_trials(nwb, 'response'))}, "
      f"no-response: {len(find_trials(nwb, 'no_response'))}")

OUTDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# =============================================================================
# 4. SELECT UNITS  (default QC, optionally AND opto-tagged, optionally AND probe)
# =============================================================================
from ephys_behavior import get_units_passed_default_qc

# Probe (device) name per unit, straight from the NWB units table. The row index
# here matches the PSTH 'unit_index' coordinate.
device_names = np.asarray(nwb.units["device_name"][:]).astype(str)
avail_probes = sorted(set(device_names))
print(f"Probes in this session: {avail_probes}")

# Base set: units passing the automated default QC (presence/isi/amplitude + not 'noise').
qc_units = np.asarray(get_units_passed_default_qc(nwb), dtype=np.int64)
sel = set(int(u) for u in qc_units)
print(f"QC-passed units: {len(sel)}")

# --- Filter 1 (optional): opto-tagged units ----------------------------------
if USE_TAGGED:
    import pandas as pd
    from ast import literal_eval
    from optical_tagging import select_tagged_units

    metrics_df = pd.read_csv(metrics_csv_path)
    tagged_df = select_tagged_units(
        metrics_df, alpha=0.05, effect_ratio=2.0, min_abs_increase=0.0,
        min_reliability=0.2, max_latency=0.006, max_jitter=0.003, correction="fdr_bh",
    )

    def _as_tuple(c):
        if isinstance(c, str):
            c = re.sub(r"np\.\w+\(([^()]*)\)", r"\1", c)
            try:
                c = literal_eval(c)
            except (ValueError, SyntaxError):
                return None
        return tuple(c) if isinstance(c, (tuple, list)) else None

    _match_idx = [i for i in range(6) if i != 2]  # match all fields except laser_name

    def _matches(c):
        c = _as_tuple(c)
        return c is not None and len(c) == 6 and all(c[i] == target_cond[i] for i in _match_idx)

    cond_mask = tagged_df["condition"].apply(_matches)
    pulse_mask = tagged_df["pulse_index"] == SELECT_PULSE_INDEX
    sigrows = tagged_df[cond_mask & pulse_mask & (tagged_df["tagged"] == True)]
    tagged_units = set(int(u) for u in sigrows["unit_id"].unique())
    print(f"Opto-tagged units ({target_cond}, pulse={SELECT_PULSE_INDEX}): {len(tagged_units)}")
    sel &= tagged_units

# --- Filter 2 (optional): specific probes ------------------------------------
if PROBES:
    want = [str(p) for p in PROBES]
    missing = [p for p in want if p not in avail_probes]
    if missing:
        print(f"WARNING: requested probes not present: {missing}")
    on_probe = set(int(u) for u in np.where(np.isin(device_names, want))[0])
    sel &= on_probe

unit_ids = sorted(sel)

# Keep only units that actually exist in the PSTH store.
try:
    psth_units = {int(u) for u in np.asarray(psth.coords["unit_index"].values)}
except Exception:
    psth_units = None
if psth_units is not None:
    unit_ids = [u for u in unit_ids if u in psth_units]

sel_probes = device_names[unit_ids] if len(unit_ids) else np.array([], dtype=str)
counts = {p: int(np.sum(sel_probes == p)) for p in sorted(set(sel_probes))}
print(f"Filters -> tagged={USE_TAGGED}, probes={'all' if not PROBES else PROBES}")
print(f"Selected units ({len(unit_ids)}): {unit_ids}")
print(f"Units per probe: {counts}")
if len(unit_ids) < 3:
    raise ValueError(
        "Fewer than 3 units after selection — a population CD needs more units. "
        "Loosen the filters or check target_cond / PROBES / metrics_csv_path."
    )

## 5. Stimulation conditions

Enumerate the distinct stimulation conditions present in the session (grouped by
laser timing/duration parameters). Each row lists the absolute `trial_ids` for
that condition, so we can later project exactly those trials onto the unstim CD.
Set `STIM_GROUP_COLS` if you want to group by different laser parameters.

In [ ]:
# =============================================================================
# 5. ENUMERATE STIMULATION CONDITIONS
# =============================================================================
from ephys_while_stimulation import summarize_laser_conditions

# Laser parameters that define a 'condition'. Adjust to taste, e.g. add
# 'laser_location', 'laser_1_power', 'laser_frequency'.
STIM_GROUP_COLS = (
    "laser_start", "laser_start_offset", "laser_end", "laser_end_offset", "laser_duration",
)

stim_summary = summarize_laser_conditions(nwb, group_cols=STIM_GROUP_COLS)
print(f"{len(stim_summary)} stimulation condition(s):")
display(stim_summary.drop(columns=["trial_ids"]))

# Map: condition index -> np.ndarray of absolute trial ids.
stim_conditions = {
    i: np.asarray(row["trial_ids"], dtype=np.int64)
    for i, row in stim_summary.iterrows()
}

## 6. Build coding directions on unstimulated trials

For each contrast in `AXES_TO_BUILD`, fit a CD using **only non-laser** trials
(class A = engaged/reward pole, class B = disengaged/no-reward pole). The axis is
cross-validated (balanced two-fold), and the fitted model additionally projects
*every* trial in the session — including the stimulation trials — onto the axis.
`d'` and `AUC` report how well the unstim axis separates its own held-out trials.

In [ ]:
# =============================================================================
# 6. BUILD CD AXES (unstimulated trials only)
# =============================================================================
from stim_axis_projection import build_axis

axes = {}
for axis_name in AXES_TO_BUILD:
    res = build_axis(
        psth, nwb, unit_ids, axis_name,
        align=ALIGN_TO, fit_window=FIT_WINDOW, proj_window=PROJ_WINDOW,
        exclude_trial_ids=opto_trial_ids,   # fit on non-laser trials only
    )
    axes[axis_name] = res
    print(f"[{axis_name}] fit on A={res.ids_a.size} vs B={res.ids_b.size} unstim trials | "
          f"d'={res.dprime:.2f}, AUC={res.auc:.2f}")

## 7. Project one stimulation condition onto the axes

Pick a stimulation condition (`STIM_CONDITION`, an index from section 5) and an
axis (`PROJECT_AXIS`). We project that condition's trials onto the unstim CD and
compare against the unstim engaged/disengaged reference distributions.

- **Strip plot** — fit-window projections for unstim-A, unstim-B and stim trials.
- **Time-resolved** — mean ± SEM projection trace for the three groups.
- **Disengagement index** — 0 at the engaged pole, 1 at the disengaged pole; a
  positive shift with small p means stimulation moves activity toward no-response.
- **Behaviour AUC** — whether the per-trial projection predicts no-response on the
  stimulated trials (the "magnitude predicts behaviour" test).

In [ ]:
# =============================================================================
# 7. PROJECT A SELECTED STIMULATION CONDITION
# =============================================================================
from stim_axis_projection import condition_shift, plot_axis_distributions, plot_axis_traces

STIM_CONDITION = 0            # index into stim_conditions (see section 5 table)
PROJECT_AXIS = "response"     # which axis in AXES_TO_BUILD to project onto

res = axes[PROJECT_AXIS]
cond_ids = stim_conditions[STIM_CONDITION]
stim_label = f"stim cond {STIM_CONDITION} (n={cond_ids.size})"
tag = f"{PROJECT_AXIS}_cond{STIM_CONDITION}"

stats = condition_shift(res, cond_ids, nwb)
print(f"Axis            : {PROJECT_AXIS}  (A={res.axis['label_a']}, B={res.axis['label_b']})")
print(f"Stim trials     : {stats['n_stim']}  ({stats['n_no_response']} no-response)")
print(f"Disengagement   : {stats['mean_di']:.3f} ± {stats['sem_di']:.3f}  "
      f"(0 = engaged pole, 1 = disengaged pole)")
print(f"Shift toward B  : Mann-Whitney p = {stats['p_toward_disengaged']:.3g}")
print(f"Behaviour AUC   : {stats['behavior_auc']:.3f}  (projection -> no-response)")

plot_axis_distributions(
    res, cond_ids, stim_label=stim_label, show=True,
    save_path=str(OUTDIR / f"projection_strip_{tag}.png"),
)
plot_axis_traces(
    res, cond_ids, stim_label=stim_label, show=True,
    save_path=str(OUTDIR / f"projection_trace_{tag}.png"),
)

## 8. Summary across all conditions and axes

Loop over every stimulation condition and every built axis, collecting the
disengagement index, the one-sided shift p-value, and the behaviour-prediction
AUC into a single table. Rows with a positive `mean_di`, small
`p_toward_disengaged`, and `behavior_auc > 0.5` are the compelling cases where
stimulation pushes the population toward the natural no-response state and that
shift tracks behaviour.

In [ ]:
# =============================================================================
# 8. SUMMARY TABLE: all conditions x all axes
# =============================================================================
import pandas as pd
from stim_axis_projection import condition_shift

rows = []
for axis_name, res in axes.items():
    for ci, cond_ids in stim_conditions.items():
        s = condition_shift(res, cond_ids, nwb)
        rows.append(dict(
            axis=axis_name,
            condition=ci,
            n_stim=s["n_stim"],
            n_no_response=s["n_no_response"],
            mean_disengagement=round(s["mean_di"], 3),
            sem_disengagement=round(s["sem_di"], 3),
            p_toward_disengaged=s["p_toward_disengaged"],
            behavior_auc=round(s["behavior_auc"], 3),
            axis_dprime=round(res.dprime, 2),
            axis_auc=round(res.auc, 2),
        ))

summary = pd.DataFrame(rows).sort_values(["axis", "condition"]).reset_index(drop=True)
summary_path = OUTDIR / "stim_cd_projection_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved summary -> {summary_path}")
display(summary)